# Credible Answers to Hard Questions: Differences-in-Differences for Natural Experiments (coding exercises - R)


## Tabla de contenidos

- [Chapter 1](#c-1)

- [Chapter 2](#c-2)

- [Chapter 3](#c-3)

- [Chapter 4](#c-4)

- [Chapter 5](#c-5)

- [Chapter 6](#c-6)

- [Chapter 7](#c-7)

- [Chapter 8](#c-8)

## Chapter 1 — Downloading and Preparing Data

This chapter covers the basic setup required to work with **all the databases in the DID Textbook**.

By the end of this chapter, the student will be able to:
- define their working folder,
- download the book's datasets,
- open any database included in the repository.

### 1.1 Working Folder

Before starting, R
 needs to know which folder to work in.

**Instruction**
Replace the path in the following command with an existing folder on your computer.

In [47]:
# Clean up the environment
rm(list = ls())

options(max.print = 1e6)

# Change the working directory
setwd("/Users/karlavega/Documents/GitHub/did_book")

### 1.2 Downloading the Data

The replication data and files are not downloaded from Stata.
Instead, all datasets are already stored locally in the project folder cloned from GitHub.
The folder structure is:

In [49]:
data_path <- "cc_xd_didtextbook_2025_9_30"


### 1.3 Loading Databases
From this point onward, the datasets used throughout the book are loaded directly from local files.
In R, the equivalent of Stata’s use command is read_dta() for Stata .dta files.
Each chapter specifies which dataset should be loaded.

In [50]:
# Load required library
library(haven)
library(dplyr)
library(stringr)
library(ggplot2)
library(broom)
library(fixest)


list.files(data_path, pattern = "\\.dta$")


character(0)

In [62]:

# -------------------------------
# Helpers
# -------------------------------

# Detecta variables tipo reltimeminus* y reltimeplus*
get_es_vars <- function(data){
  vars <- names(data)
  lead_vars <- grep("^reltimeminus", vars, value = TRUE)
  lag_vars  <- grep("^reltimeplus",  vars, value = TRUE)
  list(leads = lead_vars, lags = lag_vars)
}

# Convierte nombre "reltimeminus6" -> -6, "reltimeplus3" -> +3
term_to_reltime <- function(term){
  if (grepl("^reltimeminus", term)) return(-as.integer(sub("^reltimeminus", "", term)))
  if (grepl("^reltimeplus", term))  return( as.integer(sub("^reltimeplus",  "", term)))
  NA_integer_
}

# Saca tabla ES (coef + CI) de un modelo fixest para leads/lags
extract_es_table <- function(model, lead_vars, lag_vars){
  broom::tidy(model, conf.int = TRUE) |>
    filter(term %in% c(lead_vars, lag_vars)) |>
    mutate(rel_time = vapply(term, term_to_reltime, integer(1))) |>
    arrange(rel_time)
}

## Chapter 3 — Basic Difference in Differences

This chapter introduces the fundamental Difference-in-Differences (DiD) estimators:

- static TWFE regression,
- equivalence between TWFE and canonical DiD,
- event study and pre-trend test,
- extensions for evaluating assumptions and heterogeneity.

This chapter uses data from **Moser and Voena (2012)** to illustrate the basic Difference-in-Differences estimators.

Before starting the estimations, we load the corresponding dataset.

In [63]:
data_file <- file.path(
  "cc_xd_didtextbook_2025_9_30",
  "Data sets",
  "Moser and Voena 2012",
  "moser_voena_didtextbook.dta"
)

# Load the data
moser <- read_dta(data_file)

### 3.1 Static TWFE Regression

We estimate a model with fixed unit (`subclass`) and time (`year`) effects.
Standard errors are grouped at the `subclass` level.

In [64]:
library(fixest)

m_twfe <- feols(
  patents ~ twea | subclass + year,
  data    = moser,
  cluster = ~subclass
)

summary(m_twfe)


OLS estimation, Dep. Var.: patents
Observations: 289,920
Fixed-effects: subclass: 7,248,  year: 40
Standard-errors: Clustered (subclass) 
     Estimate Std. Error t value  Pr(>|t|)    
twea 0.288262   0.038887 7.41278 1.377e-13 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
RMSE: 0.977862     Adj. R2: 0.465316
                 Within R2: 9.571e-4

### 3.2 Equivalence between TWFE and canonical DiD

In this case, the coefficient of `twea` coincides with the classical DiD estimator.

In [65]:
# reg patents treatmentgroup post twea, cluster(subclass)
m_did <- feols(patents ~ treatmentgroup + post + twea, data = moser, cluster = ~subclass)
summary(m_did)

OLS estimation, Dep. Var.: patents
Observations: 289,920
Standard-errors: Clustered (subclass) 
                Estimate Std. Error  t value   Pr(>|t|)    
(Intercept)     0.314366   0.009819 32.01738  < 2.2e-16 ***
treatmentgroup -0.178087   0.033815 -5.26648 1.4306e-07 ***
post            0.299667   0.009061 33.07069  < 2.2e-16 ***
twea            0.288262   0.038885  7.41326 1.3721e-13 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
RMSE: 1.34503   Adj. R2: 0.013827

### 3.3 Treatment Randomization Test

This test assesses whether the treatment and control groups differ before treatment.

In [66]:
# reg patents treatmentgroup if year<=1918, cluster(subclass)
m_rand <- feols(patents ~ treatmentgroup, data = moser |> filter(year <= 1918), cluster = ~subclass)
summary(m_rand)


OLS estimation, Dep. Var.: patents
Observations: 137,712
Standard-errors: Clustered (subclass) 
                Estimate Std. Error  t value   Pr(>|t|)    
(Intercept)     0.314366   0.009819 32.01743  < 2.2e-16 ***
treatmentgroup -0.178087   0.033815 -5.26649 1.4306e-07 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
RMSE: 1.08737   Adj. R2: 0.001177

### 3.4 Event-study TWFE

Dynamic effects are estimated using leads and lags. The key test is that the pre-treatment coefficients are jointly zero.

In [67]:
es_vars <- get_es_vars(moser)
lead_vars <- es_vars$leads
lag_vars  <- es_vars$lags

if(length(c(lead_vars, lag_vars)) == 0){
  stop("No encuentro variables reltimeminus* / reltimeplus* en df. Revisa nombres/creación.")
}



In [71]:
# En R, eso es: patents ~ treatmentgroup + leads + lags + FE(year)
fml_es <- as.formula(paste0(
  "patents ~ treatmentgroup + ",
  paste(c(lead_vars, lag_vars), collapse = " + "),
  " | year"
))

m_es <- feols(fml_es, data = moser, cluster = ~subclass)
summary(m_es)

OLS estimation, Dep. Var.: patents
Observations: 289,920
Fixed-effects: year: 40
Standard-errors: Clustered (subclass) 
                Estimate Std. Error   t value   Pr(>|t|)    
treatmentgroup -0.190807   0.060313 -3.163589 1.5648e-03 ** 
reltimeminus1  -0.027034   0.044596 -0.606192 5.4441e-01    
reltimeminus2  -0.096375   0.036719 -2.624668 8.6915e-03 ** 
reltimeminus3  -0.063575   0.034334 -1.851672 6.4114e-02 .  
reltimeminus4   0.023706   0.039432  0.601185 5.4774e-01    
reltimeminus5   0.068700   0.046862  1.466006 1.4269e-01    
reltimeminus6   0.013538   0.038389  0.352640 7.2437e-01    
reltimeminus7   0.024781   0.050856  0.487277 6.2608e-01    
reltimeminus8   0.038690   0.049034  0.789059 4.3010e-01    
reltimeminus9   0.005911   0.042835  0.137994 8.9025e-01    
reltimeminus10  0.048198   0.048260  0.998713 3.1797e-01    
reltimeminus11 -0.066179   0.056872 -1.163642 2.4461e-01    
reltimeminus12  0.029927   0.042829  0.698769 4.8472e-01    
reltimeminus13  0.052931  

In [73]:
# Test of pre-trends: test reltimeminus1 ... reltimeminus18
# En R: wald joint test
pretest_vars <- intersect(paste0("reltimeminus", 1:18), names(coef(m_es)))
if(length(pretest_vars) > 0){
  wald(m_es, pretest_vars)   # H0: todos esos coef = 0
}

# ES plot (equivalente a tu bloque de matrices + twoway)
es_tab <- extract_es_table(m_es, lead_vars, lag_vars)

p_es1 <- ggplot(es_tab, aes(x = rel_time, y = estimate)) +
  geom_point() +
  geom_line() +
  geom_errorbar(aes(ymin = conf.low, ymax = conf.high), width = 0) +
  labs(
    title = "TWFE Event-study estimates",
    x = "Relative time to year before TWEA",
    y = "Effect"
  ) +
  scale_x_continuous(breaks = seq(min(es_tab$rel_time, na.rm = TRUE),
                                  max(es_tab$rel_time, na.rm = TRUE), by = 3)) +
  coord_cartesian(ylim = c(-0.25, 1))

# ggsave("graphES_moser1_R.pdf", p_es1, width = 7.5, height = 5)

# Verifying by hand DID: m1-m2-(m3-m4)
m1 <- moser |> filter(year == 1919, treatmentgroup == 1) |> summarise(m = mean(patents, na.rm=TRUE)) |> pull(m)
m2 <- moser |> filter(year == 1918, treatmentgroup == 1) |> summarise(m = mean(patents, na.rm=TRUE)) |> pull(m)
m3 <- moser |> filter(year == 1919, treatmentgroup == 0) |> summarise(m = mean(patents, na.rm=TRUE)) |> pull(m)
m4 <- moser |> filter(year == 1918, treatmentgroup == 0) |> summarise(m = mean(patents, na.rm=TRUE)) |> pull(m)
did_hand <- (m1 - m2) - (m3 - m4)
did_hand

Wald test, H0: joint nullity of reltimeminus1, reltimeminus2, reltimeminus3, reltimeminus4, reltimeminus5, reltimeminus6 and 12 others
 stat = 3.79168, p-value = 8.936e-8, on 18 and 289,840 DoF, VCOV: Clustered (subclass).

[1] 0.02352017

### 3.5 Event-study without pre-treatment periods

The model is re-estimated excluding the leads, allowing for comparison with the dynamic DiD in the post-treatment period.


In [77]:
if(!("yearpost" %in% names(moser))) stop("No existe yearpost en df (necesaria para el ES post-only).")

fml_es_post <- as.formula(paste0(
  "patents ~ treatmentgroup + ",
  paste(lag_vars, collapse = " + "),
  " | yearpost"
))

m_es_post <- feols(fml_es_post, data = moser, cluster = ~subclass)
summary(m_es_post)

es_post_tab <- broom::tidy(m_es_post, conf.int = TRUE) |>
  filter(term %in% lag_vars) |>
  mutate(rel_time = as.integer(sub("^reltimeplus", "", term))) |>
  arrange(rel_time)

# Overlay plot: Without Pre-Periods vs With Pre-Periods (como tu twoway)
# Para superponer, armamos un "res_post" del modelo completo pero SOLO post-periodos:
es_full_post <- es_tab |> filter(rel_time >= 0) |>
  mutate(series = "With Pre-Periods")
es_post_only <- es_post_tab |>
  mutate(series = "Without Pre-Periods")

p_es2 <- ggplot() +
  geom_line(data = es_post_only, aes(rel_time, estimate, linetype = series)) +
  geom_point(data = es_post_only, aes(rel_time, estimate, shape = series)) +
  geom_errorbar(data = es_post_only, aes(rel_time, ymin = conf.low, ymax = conf.high), width = 0) +
  geom_line(data = es_full_post, aes(rel_time, estimate, linetype = series)) +
  geom_point(data = es_full_post, aes(rel_time, estimate, shape = series)) +
  geom_errorbar(data = es_full_post, aes(rel_time, ymin = conf.low, ymax = conf.high), width = 0) +
  labs(
    title = "TWFE Event-study estimates",
    x = "Relative time to year before TWEA",
    y = "Effect",
    linetype = NULL, shape = NULL
  ) +
  scale_x_continuous(breaks = seq(0, max(es_full_post$rel_time, na.rm = TRUE), by = 3)) +
  coord_cartesian(ylim = c(-0.25, 1))

# ggsave("graphES_moser2_R.pdf", p_es2, width = 7.5, height = 5)

# did_imputation (Borusyak et al) — equivalente recomendado en R
# Stata: gen cohort=1919 if treatmentgroup==1 ; did_imputation patents subclass year cohort ...
df <- moser |> mutate(cohort = ifelse(treatmentgroup == 1, 1919L, NA_integer_))

# TODO (mejor opción en R, 2 rutas):
# (A) fixest::sunab(cohort, year) si lo que quieres es un ES robusto tipo Sun & Abraham:
# m_sunab <- feols(patents ~ sunab(cohort, year) | subclass + year, data = df, cluster = ~subclass)
#
# (B) paquete "didimputation" (si buscas el estimador Borusyak-Gardner exacto en R):
# remotes::install_github("kylebutts/didimputation")  # ejemplo, depende de tu preferencia
# library(didimputation)

# Equivalent to having all year FEs in regression:
# Stata: reg patents i.year treatmentgroup reltimeplus*, cluster(subclass)
fml_post_all_yearfe <- as.formula(paste0(
  "patents ~ treatmentgroup + ",
  paste(lag_vars, collapse = " + "),
  " | year"
))
m_post_all_yearfe <- feols(fml_post_all_yearfe, data = moser, cluster = ~subclass)
summary(m_post_all_yearfe)

OLS estimation, Dep. Var.: patents
Observations: 289,920
Fixed-effects: yearpost: 22
Standard-errors: Clustered (subclass) 
                Estimate Std. Error   t value   Pr(>|t|)    
treatmentgroup -0.178087   0.033818 -5.266118 1.4334e-07 ***
reltimeplus1    0.010801   0.030976  0.348674 7.2734e-01    
reltimeplus2    0.039302   0.030205  1.301183 1.9324e-01    
reltimeplus3    0.049842   0.035973  1.385567 1.6592e-01    
reltimeplus4    0.016650   0.041810  0.398224 6.9048e-01    
reltimeplus5    0.080307   0.049547  1.620826 1.0510e-01    
reltimeplus6    0.008982   0.041055  0.218777 8.2683e-01    
reltimeplus7    0.012950   0.047263  0.274000 7.8409e-01    
reltimeplus8    0.088554   0.043202  2.049751 4.0425e-02 *  
reltimeplus9    0.188545   0.047944  3.932599 8.4816e-05 ***
reltimeplus10   0.211549   0.050361  4.200649 2.6934e-05 ***
reltimeplus11   0.119680   0.052021  2.300595 2.1443e-02 *  
reltimeplus12   0.110152   0.059302  1.857459 6.3286e-02 .  
reltimeplus13   0.2266

OLS estimation, Dep. Var.: patents
Observations: 289,920
Fixed-effects: year: 40
Standard-errors: Clustered (subclass) 
                Estimate Std. Error   t value   Pr(>|t|)    
treatmentgroup -0.178087   0.033819 -5.265955 1.4347e-07 ***
reltimeplus1    0.010801   0.030977  0.348664 7.2735e-01    
reltimeplus2    0.039302   0.030206  1.301143 1.9325e-01    
reltimeplus3    0.049842   0.035974  1.385524 1.6593e-01    
reltimeplus4    0.016650   0.041811  0.398212 6.9049e-01    
reltimeplus5    0.080307   0.049549  1.620775 1.0511e-01    
reltimeplus6    0.008982   0.041056  0.218770 8.2684e-01    
reltimeplus7    0.012950   0.047265  0.273991 7.8410e-01    
reltimeplus8    0.088554   0.043203  2.049687 4.0431e-02 *  
reltimeplus9    0.188545   0.047946  3.932477 8.4859e-05 ***
reltimeplus10   0.211549   0.050363  4.200519 2.6950e-05 ***
reltimeplus11   0.119680   0.052023  2.300524 2.1447e-02 *  
reltimeplus12   0.110152   0.059304  1.857402 6.3295e-02 .  
reltimeplus13   0.226678  

### 3.6 Undetected Linear Pretrends

This section assesses whether the results could be explained by differential linear trends not detected by standard tests.

In [78]:
df_pt <- moser |> filter(year >= 1912, year <= 1939)

# Limitar a 6 pre-periodos
K <- 6
lead6 <- intersect(paste0("reltimeminus", 1:K), names(df_pt))
lag21 <- intersect(paste0("reltimeplus", 1:21), names(df_pt))  # en tu Stata usas 21 post

if(length(lead6) < K) stop("No tengo los 6 leads reltimeminus1..6 en df_pt.")
if(length(lag21) == 0) stop("No encuentro reltimeplus* en df_pt.")

fml_pt <- as.formula(paste0(
  "patents ~ ", paste(c(lead6, lag21), collapse = " + "),
  " | treatmentgroup + year"
))

m_pt <- feols(fml_pt, data = df_pt, cluster = ~subclass)
summary(m_pt)

# Extraer betahat y Sigma (clusterizada) para los 6 pre
b_pre <- coef(m_pt)[lead6]
V_pre <- vcov(m_pt, cluster = ~subclass)[lead6, lead6, drop = FALSE]

t_pre <- -as.integer(sub("^reltimeminus", "", lead6))

# Equivalente a "pretrends power 0.5, numpre(6)"
slope <- pretrends::slope_for_power(
  sigma           = V_pre,
  targetPower     = 0.5,
  tVec            = t_pre,
  referencePeriod = 0
)
slope

# Plot ES (solo -6..21) con línea y = slope * x (como tu function y=x*slope)
m_es_short <- feols(
  as.formula(paste0(
    "patents ~ treatmentgroup + ", paste(c(lead6, lag21), collapse = " + "), " | year"
  )),
  data = df_pt, cluster = ~subclass
)

es_short <- extract_es_table(m_es_short, lead6, lag21)

p_es3 <- ggplot(es_short, aes(rel_time, estimate)) +
  geom_point() + geom_line() +
  geom_errorbar(aes(ymin = conf.low, ymax = conf.high), width = 0) +
  geom_abline(intercept = 0, slope = as.numeric(slope), linetype = "dashed") +
  labs(
    title = "TWFE Event-study estimates",
    x = "Relative time to year before TWEA",
    y = "Effect"
  ) +
  scale_x_continuous(breaks = seq(-6, 21, by = 3)) +
  coord_cartesian(ylim = c(-0.25, 1))

# ggsave("graphES_moser3_R.pdf", p_es3, width = 7.5, height = 5)

OLS estimation, Dep. Var.: patents
Observations: 202,944
Fixed-effects: treatmentgroup: 2,  year: 28
Standard-errors: Clustered (subclass) 
               Estimate Std. Error   t value   Pr(>|t|)    
reltimeminus1 -0.027034   0.044596 -0.606192 5.4441e-01    
reltimeminus2 -0.096375   0.036719 -2.624670 8.6915e-03 ** 
reltimeminus3 -0.063575   0.034334 -1.851674 6.4113e-02 .  
reltimeminus4  0.023706   0.039432  0.601185 5.4774e-01    
reltimeminus5  0.068700   0.046862  1.466007 1.4269e-01    
reltimeminus6  0.013538   0.038389  0.352641 7.2437e-01    
reltimeplus1   0.023520   0.044417  0.529532 5.9645e-01    
reltimeplus2   0.052021   0.044239  1.175908 2.3967e-01    
reltimeplus3   0.062562   0.042763  1.462985 1.4351e-01    
reltimeplus4   0.029369   0.053509  0.548865 5.8312e-01    
reltimeplus5   0.093027   0.063435  1.466484 1.4256e-01    
reltimeplus6   0.021701   0.050222  0.432112 6.6567e-01    
reltimeplus7   0.025670   0.060922  0.421354 6.7351e-01    
reltimeplus8   0.101

[1] 0.01012984

### 3.7 Variance of the long-term effect

The variance of the results between groups is compared 14 years after treatment.

In [79]:
if(!("diffpatentswrt1918" %in% names(moser))) stop("No existe diffpatentswrt1918 en df.")

df_1932 <- moser |> filter(year == 1932)

x_ctrl  <- df_1932 |> filter(treatmentgroup == 0) |> pull(diffpatentswrt1918)
x_treat <- df_1932 |> filter(treatmentgroup == 1) |> pull(diffpatentswrt1918)

# sdtest ~ test de igualdad de varianzas (F-test) en R:
vt_1932 <- var.test(x_treat, x_ctrl)
vt_1932

sd_ctrl  <- sd(x_ctrl,  na.rm = TRUE)
sd_treat <- sd(x_treat, na.rm = TRUE)
sd_effects <- sd_treat - sd_ctrl
sd_effects

# Point estimate of treatment effect at 1932:
m_te_1932 <- lm(diffpatentswrt1918 ~ treatmentgroup, data = df_1932)
summary(m_te_1932)

b_hat <- coef(m_te_1932)[["treatmentgroup"]]
ci_custom <- c(b_hat - 1.96 * sd_effects, b_hat + 1.96 * sd_effects)
ci_custom



	F test to compare two variances

data:  x_treat and x_ctrl
F = 1.2879, num df = 335, denom df = 6911, p-value = 0.0008145
alternative hypothesis: true ratio of variances is not equal to 1
95 percent confidence interval:
 1.108809 1.512649
sample estimates:
ratio of variances 
          1.287929 


[1] 0.2390978


Call:
lm(formula = diffpatentswrt1918 ~ treatmentgroup, data = df_1932)

Residuals:
     Min       1Q   Median       3Q      Max 
-31.3578  -0.3578  -0.3578   0.6422  27.6422 

Coefficients:
               Estimate Std. Error t value Pr(>|t|)    
(Intercept)     0.35778    0.02147  16.668  < 2e-16 ***
treatmentgroup  0.64222    0.09969   6.442 1.26e-10 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 1.785 on 7246 degrees of freedom
Multiple R-squared:  0.005694,	Adjusted R-squared:  0.005557 
F-statistic:  41.5 on 1 and 7246 DF,  p-value: 1.256e-10


[1] 0.1735848 1.1108481

### 3.8 Placebo tests

Placebo tests are performed to assess spurious heterogeneity before treatment.

In [80]:
years <- 1900:1939

placebo <- lapply(years, function(yy){
  dyy <- moser |> filter(year == yy)
  x0 <- dyy |> filter(treatmentgroup == 0) |> pull(diffpatentswrt1918)
  x1 <- dyy |> filter(treatmentgroup == 1) |> pull(diffpatentswrt1918)

  # Si algún año no tiene datos suficientes:
  if(length(na.omit(x0)) < 2 || length(na.omit(x1)) < 2){
    return(data.frame(year = yy, p_value = NA_real_, sd_diff = NA_real_))
  }

  vt <- var.test(x1, x0)
  sd_diff <- sd(x1, na.rm=TRUE) - sd(x0, na.rm=TRUE)

  data.frame(year = yy, p_value = vt$p.value, sd_diff = sd_diff)
})

placebo_df <- bind_rows(placebo)
placebo_df

year,p_value,sd_diff
<int>,<dbl>,<dbl>
1900,1.068152e-36,-0.61160643
1901,8.517629e-25,-0.51625837
1902,3.651660e-38,-0.62722211
1903,3.210358e-35,-0.59126117
1904,4.113821e-48,-0.68834581
1905,1.037844e-46,-0.67754075
1906,2.913373e-41,-0.64710256
1907,3.621998e-11,-0.33130660
1908,5.485717e-25,-0.49884305


## Chapter 4 — Extensions to the Basic DiD Model

This chapter extends the basic DiD estimators to address:

- baseline covariate control,
- interactive fixed effects,
- synthetic control methods,
- sensitivity analysis to pretrend violations.

### 4.1 Estimators with Controls

Stata absorbs `year#patents1900` (time-varying slope on a baseline covariate). In R, the most direct equivalent is interacting the baseline covariate with year using `i(year, patents1900)`.


In [ ]:
if("patents1900" %in% names(moser)){
  f_ctrl <- as.formula(
    paste0(
      "patents ~ ", paste(c(lead_vars, lag_vars), collapse = " + "),
      " + i(year, patents1900) | treatmentgroup + year"
    )
  )

  m_ctrl <- feols(f_ctrl, data = moser, cluster = ~subclass)
  summary(m_ctrl)

  if(length(lead_vars) > 0) wald(m_ctrl, lead_vars)

  plot_es_fixest(m_ctrl, title = "Event-study with baseline patents control")
}

### 4.2 Interactive Fixed Effects

Best R equivalent is the **fect** package (Xu) which implements interactive fixed effects / generalized synthetic control and related estimators.


In [ ]:
library(fect)

# Stata: fect patents, treat(twea) unit(subclass) time(year) method("ife") r(4) cv
# In R, specify index = c("unit","time") and the treatment variable.

moser_df <- as.data.frame(moser)

# Cross-validate number of factors (rough analogue)
fit_cv <- fect(patents ~ twea,
               data = moser_df,
               index = c("subclass","year"),
               method = "ife",
               r = 4,
               CV = TRUE)
summary(fit_cv)

set.seed(1)
fit_ife <- fect(patents ~ twea,
                data = moser_df,
                index = c("subclass","year"),
                method = "ife",
                r = 2,
                se = TRUE)
summary(fit_ife)
plot(fit_ife)

### 4.3 Synthetic Control

Stata uses `sdid_event ... method("sc")`. In R, the closest "classic" synthetic control workflow is the **Synth** package (Abadie et al.). This requires defining a single treated unit (or aggregating treated units) and specifying pre-period predictors.


The exercise is repeated after removing the pre-treatment average per unit.

### 4.4 Synthetic DiD

DiD and synthetic control are combined into a single estimator.

### 4.5 Sensitivity Analysis (Rambachan and Roth)

The robustness of the estimated effects is evaluated in the face of potential violations of pre-trends.

Sensitivity using only a specific lead and lag.

## Chapter 5 — TWFE and Weight Decomposition

This chapter examines how TWFE estimators combine comparisons between units and periods, using data from **Gentzkow et al. (2011)**.

Before we begin, we load the database corresponding to this chapter.

### 5.1 TWFE Regression

A standard TWFE model is estimated and the estimator is decomposed into its implicit weights.

### 5.2 TWFE with State-Specific Trends

Each state is allowed to have its own temporal trend, controlling for unobserved heterogeneity.

### 5.3 First Difference Regression

The model is estimated using first differences with specific trends, and the estimator weights are re-analyzed.

## Chapter 6 — Multiple Cohorts and Alternative Estimators

This chapter analyzes data with **stepwise treatments** and compares different Difference-in-Differences estimators proposed in the recent literature.

Before starting, the database used in this chapter is loaded.

### 6.1 Static TWFE Regression

A standard TWFE model with fixed effects per state and year is estimated,
weighted by state population.

### 6.2 Decomposition of the TWFE estimator

The implicit weights of the TWFE estimator are analyzed and it is assessed whether they are correlated with the duration of exposure.

### 6.3 Test of randomization in treatment timing

This assesses whether the timing of treatment initiation can be considered exogenous.

### 6.4 Event-study TWFE

Dynamic effects are estimated using an event-study TWFE and pre-trends are tested.

### 6.5 Decomposition of the First Dynamic Effect

The first coefficient of the event study is decomposed
to analyze which comparisons identify it.

### 6.6 Sun and Abraham (2021) Estimator

A robust event study is estimated for stepped treatments using adoption cohorts.

### 6.7 Callaway and Sant’Anna (2021) Estimator

Dynamic average effects are estimated by cohort using untreated groups as controls.

### 6.8 Estimador de de Chaisemartin y D’Haultfoeuille

Se estiman efectos dinámicos permitiendo heterogeneidad entre cohortes y a lo largo del tiempo.

### 6.9 Estimador de Borusyak et al. (2021)

Se estima el efecto dinámico usando imputación bajo supuestos de tendencias paralelas.


## Chapter 7 — Continuous Treatments and Robustness Tests

This chapter analyzes DiD estimators with continuous treatment (ntrgap) and studies various pre-trend and robustness tests, using data from Pierce and Schott (2016).

Before starting, the database corresponding to this chapter is loaded.

### 7.1 TWFE Regressions

Simple TWFE regressions are estimated for different years, treating the NTR gap as a continuous treatment.

### 7.2 Weight Analysis

The implicit weights of the estimator are analyzed when the treatment is continuous.

### 7.3 Treatment Randomization Test

This test assesses whether the NTR gap is correlated with pre-treatment characteristics.


### 7.4 Stute Test

The non-parametric Stute test is implemented to evaluate the validity of the DiD design.

A joint post-treatment effect test is performed.

Se evalúa la presencia de *quasi-stayers* en la intensidad del tratamiento.


### 7.5 Pre-trend (linear) tests

Differences in trends are evaluated before treatment.


### 7.6 Pre-trends with industry-specific trends

Industry-specific linear trends are permitted, both in parametric and non-parametric form.

Joint test of pre-trends with specific trends.

### 7.7 Stute Test with Linear Trends (Post)

The robustness of the post-treatment effects is evaluated, allowing for linear trends.

Joint test of post-treatment effects with linear trends.

### 7.8 Estimators with Linear Trends

Effects are estimated by explicitly allowing
linear trends by industry.

## Chapter 8 — Dynamics with Lagged Treatments

This chapter examines DiD models with **dynamic treatments** and **lagged effects**, again using data from **Gentzkow et al. (2011)**.

The previously used database is reloaded to ensure a clean environment before estimations.

### 8.1 Is the change in newspapers as-good-as-random?

We evaluate whether the change in the number of newspapers can be considered exogenous, conditional on lagged variables.

### 8.2 TWFE with lagged treatments

A TWFE model is estimated that includes both the contemporaneous treatment and its lag.

### 8.3 Unnormalized Event Study

Unnormalized dynamic effects are estimated using the Chaisemartin and D’Haultfoeuille estimator.

### 8.4 Trajectory Decomposition

The individual trajectories that make up the unnormalized average effects are analyzed.

### 8.5 Normalized Event Study

Dynamic effects are normalized to facilitate interpretation and comparison between periods.

### 8.6 Lagged Effects of Treatment Test

This test assesses whether treatment lags affect the outcome.

### 8.7 Estimators without lagged treatment effects

Models are estimated that assume the absence of effects of past treatments on the current outcome.